# Домашнее задание № 2. Мешок слов

## Задание 1 (3 балла)

У векторайзеров в sklearn есть встроенная токенизация на регулярных выражениях. Найдите способо заменить её на кастомную токенизацию

Обучите векторайзер с дефолтной токенизацией и с токенизацией razdel.tokenize. Обучите классификатор (любой) с каждым из векторизаторов. Сравните метрики и выберете победителя.

(в вашей тетрадке должен быть код обучения и все метрики; если вы сдаете в .py файлах то сохраните полученные метрики в отдельном файле или в комментариях)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

In [5]:
!pip install razdel

In [6]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score
from razdel import tokenize
import re

In [7]:
data = pd.read_csv('/content/drive/MyDrive/labeled.csv', sep=',')

In [8]:
data

,comment,toxic
0,"Верблюдов-то за что? Дебилы, бл...\n",1.0
1,"Хохлы, это отдушина затюканого россиянина, мол...",1.0
2,Собаке - собачья смерть\n,1.0
3,"Страницу обнови, дебил. Это тоже не оскорблени...",1.0
4,"тебя не убедил 6-страничный пдф в том, что Скр...",1.0
...,...,...
14407,Вонючий совковый скот прибежал и ноет. А вот и...,1.0
14408,А кого любить? Гоблина тупорылого что-ли? Или ...,1.0
14409,"Посмотрел Утомленных солнцем 2. И оказалось, ч...",0.0
14410,КРЫМОТРЕД НАРУШАЕТ ПРАВИЛА РАЗДЕЛА Т.К В НЕМ Н...,1.0


In [9]:
print("Размер датасета:", data.shape)
print("Распределение меток:")
print(data['toxic'].value_counts())

Размер датасета: (14412, 2)
Распределение меток:
toxic
0.0    9586
1.0    4826
Name: count, dtype: int64


In [ ]:
X = data['comment']
y = data['toxic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Тренировочная выборка: {X_train.shape[0]} примеров")
print(f"Тестовая выборка: {X_test.shape[0]} примеров")

Тренировочная выборка: 11529 примеров
Тестовая выборка: 2883 примеров


In [ ]:
# 1. Векторайзер со стандартной токенизацией sklearn
print("\n" + "="*50)
print("1. ВЕКТОРАЙЗЕР СО СТАНДАРТНОЙ ТОКЕНИЗАЦИЕЙ")
print("="*50)

default_vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.9,
    ngram_range=(1, 2)
)

# Обучение векторайзера и преобразование текстов
X_train_default = default_vectorizer.fit_transform(X_train)
X_test_default = default_vectorizer.transform(X_test)

print(f"Размерность признаков: {X_train_default.shape}")


1. ВЕКТОРАЙЗЕР СО СТАНДАРТНОЙ ТОКЕНИЗАЦИЕЙ
Размерность признаков: (11529, 5000)


In [ ]:
# Обучение классификатора
default_classifier = LogisticRegression(
    random_state=42,
    max_iter=1000
)
default_classifier.fit(X_train_default, y_train)

# Предсказания и метрики
y_pred_default = default_classifier.predict(X_test_default)

accuracy_default = accuracy_score(y_test, y_pred_default)
f1_default = f1_score(y_test, y_pred_default)

print(f"Accuracy: {accuracy_default:.4f}")
print(f"F1-score: {f1_default:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_default))

Accuracy: 0.8231
F1-score: 0.6824

Classification Report:
              precision    recall  f1-score   support

         0.0       0.81      0.95      0.88      1918
         1.0       0.85      0.57      0.68       965

    accuracy                           0.82      2883
   macro avg       0.83      0.76      0.78      2883
weighted avg       0.83      0.82      0.81      2883



In [ ]:
# 2. Векторайзер с кастомной токенизацией через razdel
print("\n" + "="*50)
print("2. ВЕКТОРАЙЗЕР С ТОКЕНИЗАЦИЕЙ RAZDEL")
print("="*50)

def razdel_tokenizer(text):
    """Кастомная токенизация с использованием razdel"""
    # Приводим к строке на случай NaN значений
    text = str(text) if not pd.isna(text) else ""
    # Токенизируем и извлекаем текст токенов
    tokens = [token.text for token in tokenize(text)]
    return tokens


# Создаем векторайзер с кастомной токенизацией
razdel_vectorizer = TfidfVectorizer(
    tokenizer=razdel_tokenizer,
    max_features=5000,
    min_df=2,
    max_df=0.9,
    ngram_range=(1, 2)
)

# Обучение векторайзера и преобразование текстов
X_train_razdel = razdel_vectorizer.fit_transform(X_train)
X_test_razdel = razdel_vectorizer.transform(X_test)

print(f"Размерность признаков: {X_train_razdel.shape}")

# Обучение классификатора
razdel_classifier = LogisticRegression(
    random_state=42,
    max_iter=1000
)
razdel_classifier.fit(X_train_razdel, y_train)

# Предсказания и метрики
y_pred_razdel = razdel_classifier.predict(X_test_razdel)

accuracy_razdel = accuracy_score(y_test, y_pred_razdel)
f1_razdel = f1_score(y_test, y_pred_razdel)

print(f"Accuracy: {accuracy_razdel:.4f}")
print(f"F1-score: {f1_razdel:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_razdel))


2. ВЕКТОРАЙЗЕР С ТОКЕНИЗАЦИЕЙ RAZDEL


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Размерность признаков: (11529, 5000)
Accuracy: 0.8283
F1-score: 0.6972

Classification Report:
              precision    recall  f1-score   support

         0.0       0.82      0.95      0.88      1918
         1.0       0.85      0.59      0.70       965

    accuracy                           0.83      2883
   macro avg       0.84      0.77      0.79      2883
weighted avg       0.83      0.83      0.82      2883



In [ ]:
# 3. Сравнение результатов
print("\n" + "="*50)
print("3. СРАВНЕНИЕ РЕЗУЛЬТАТОВ")
print("="*50)

print(f"{'Метрика':<15} {'Стандартная':<12} {'Razdel':<12} {'Победитель':<10}")
print("-" * 50)
print(f"{'Accuracy':<15} {accuracy_default:.4f}      {accuracy_razdel:.4f}      {'Razdel' if accuracy_razdel > accuracy_default else 'Стандартная'}")
print(f"{'F1-score':<15} {f1_default:.4f}      {f1_razdel:.4f}      {'Razdel' if f1_razdel > f1_default else 'Стандартная'}")

# Определяем победителя
if accuracy_razdel > accuracy_default and f1_razdel > f1_default:
    winner = "RAZDEL TOKENIZATION"
elif accuracy_default > accuracy_razdel and f1_default > f1_razdel:
    winner = "STANDARD TOKENIZATION"
else:
    winner = "НЕТ ЯВНОГО ПОБЕДИТЕЛЯ"

print(f"\n🏆 ПОБЕДИТЕЛЬ: {winner}")

# 4. Демонстрация разницы в токенизации
print("\n" + "="*50)
print("4. ДЕМОНСТРАЦИЯ РАЗЛИЧИЙ В ТОКЕНИЗАЦИИ")
print("="*50)

sample_text = "Привет, как дела? Это тестовый текст-пример!"
print(f"Исходный текст: '{sample_text}'")
print(f"Стандартная токенизация: {default_vectorizer.build_tokenizer()(sample_text)}")
print(f"Razdel токенизация: {razdel_tokenizer(sample_text)}")


3. СРАВНЕНИЕ РЕЗУЛЬТАТОВ
Метрика         Стандартная  Razdel       Победитель
--------------------------------------------------
Accuracy        0.8231      0.8283      Razdel
F1-score        0.6824      0.6972      Razdel

🏆 ПОБЕДИТЕЛЬ: RAZDEL TOKENIZATION

4. ДЕМОНСТРАЦИЯ РАЗЛИЧИЙ В ТОКЕНИЗАЦИИ
Исходный текст: 'Привет, как дела? Это тестовый текст-пример!'
Стандартная токенизация: ['Привет', 'как', 'дела', 'Это', 'тестовый', 'текст', 'пример']
Razdel токенизация: ['Привет', ',', 'как', 'дела', '?', 'Это', 'тестовый', 'текст-пример', '!']


## Задание 2 (3 балла)

Обучите 2 любых разных классификатора из семинара. Предскажите токсичность для текстов из тестовой выборки (используйте одну и ту же выборку для обоих классификаторов) и найдите 10 самых токсичных для каждого из классификаторов. Сравните получаемые тексты - какие тексты совпадают, какие отличаются, правда ли тексты токсичные?

Требования к моделям:   
а) один классификатор должен использовать CountVectorizer, другой TfidfVectorizer  
б) у векторазера должны быть вручную заданы как минимум 5 параметров (можно ставить разные параметры tfidfvectorizer и countvectorizer)  
в) у классификатора должно быть задано вручную как минимум 2 параметра (по возможности)  
г)  f1 мера каждого из классификаторов должна быть минимум 0.75  

*random_seed не считается за параметр

In [19]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import re

# Загрузка данных
data = pd.read_csv('/content/drive/MyDrive/labeled.csv', sep=',')
print(f"Размер датасета: {len(data)}")
print(f"Баланс классов:\n{data['toxic'].value_counts()}")

russian_stopwords = [
    'и', 'в', 'на', 'с', 'по', 'у', 'о', 'к', 'а', 'но', 'это', 'вот',
    'ну', 'да', 'нет', 'же', 'ли', 'бы', 'то', 'меня', 'тебя', 'он',
    'она', 'они', 'мы', 'вы', 'мне', 'тебе', 'ему', 'ей', 'нам', 'вам',
    'мной', 'тобой', 'им', 'ею', 'нами', 'вами', 'мой', 'твой', 'его',
    'её', 'наш', 'ваш', 'их', 'свой', 'кто', 'что', 'как', 'где', 'когда',
    'почему', 'зачем', 'какой', 'который', 'сколько', 'чей', 'чтобы',
    'если', 'хотя', 'потому', 'так', 'тоже', 'или', 'ибо', 'чем', 'не',
    'ни', 'без', 'до', 'из', 'от', 'при', 'через', 'для', 'про', 'со',
    'тот', 'этот', 'такой', 'какой-то', 'кое-какой', 'какой-нибудь',
    'некий', 'некоторый', 'всякий', 'каждый', 'любой', 'самый', 'другой',
    'иной', 'сам', 'сама', 'само', 'сами', 'весь', 'вся', 'всё', 'все',
    'самого', 'самой', 'самом', 'самих', 'всего', 'всей', 'всём', 'всех',
    'есть', 'можно', 'очень', 'например', 'было'
]

# Улучшенная предобработка текста
def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'[^\w\s!?]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

data['comment_clean'] = data['comment'].apply(preprocess_text)

# Разделение на train/test
X = data['comment_clean']
y = data['toxic']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nРазмер тренировочной выборки: {len(X_train)}")
print(f"Размер тестовой выборки: {len(X_test)}")
print(f"Баланс классов: 0 - {y_train.value_counts()[0]}, 1 - {y_train.value_counts()[1]}")

# ПАРА 1: LogisticRegression + CountVectorizer
print("\n" + "="*70)
print("ПАРА 1: LogisticRegression + CountVectorizer")
print("="*70)

count_vectorizer = CountVectorizer(
    max_features=12000,
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 3),
    analyzer='word',
    binary=True,
)

X_train_count = count_vectorizer.fit_transform(X_train)
X_test_count = count_vectorizer.transform(X_test)

# LogisticRegression
logreg = LogisticRegression(
    C=0.8,
    class_weight={0: 1, 1: 2},
    max_iter=2000,
    solver='liblinear',
    random_state=42,
    penalty='l2'
)

logreg.fit(X_train_count, y_train)
y_pred_logreg = logreg.predict(X_test_count)
f1_logreg = f1_score(y_test, y_pred_logreg)

print(f"F1-score LogisticRegression: {f1_logreg:.4f}")
print(classification_report(y_test, y_pred_logreg))

# ПАРА 2: RandomForest + TfidfVectorizer
print("\n" + "="*70)
print("ПАРА 2: RandomForest + TfidfVectorizer")
print("="*70)

tfidf_vectorizer = TfidfVectorizer(
    max_features=100000,
    min_df=2,
    max_df=0.75,
    ngram_range=(1, 5),
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=True,
    analyzer='word',
    norm='l2',
    stop_words=russian_stopwords,
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# RandomForest
rf = RandomForestClassifier(
    n_estimators=1500,
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight={0: 1, 1: 2},
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_tfidf, y_train)
y_pred_rf = rf.predict(X_test_tfidf)
f1_rf = f1_score(y_test, y_pred_rf)

print(f"F1-score RandomForest: {f1_rf:.4f}")
print(classification_report(y_test, y_pred_rf))

# ФИНАЛЬНАЯ ПРОВЕРКА
print("\n" + "="*70)
print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ")
print("="*70)

print(f"LogisticRegression + CountVectorizer: F1 = {f1_logreg:.4f}")
print(f"RandomForest + TfidfVectorizer: F1 = {f1_rf:.4f}")


# АНАЛИЗ САМЫХ ТОКСИЧНЫХ ТЕКСТОВ
print("\n" + "="*70)
print("АНАЛИЗ САМЫХ ТОКСИЧНЫХ ТЕКСТОВ")
print("="*70)

# Получаем вероятности
test_probs_logreg = logreg.predict_proba(X_test_count)[:, 1]
test_probs_rf = rf.predict_proba(X_test_tfidf)[:, 1]

# Создаем DataFrame с результатами
results_df = pd.DataFrame({
    'text': X_test.values,
    'text_clean': X_test,
    'toxic_actual': y_test,
    'prob_logreg': test_probs_logreg,
    'prob_rf': test_probs_rf
})

# Топ-10 самых токсичных по каждому классификатору
top_toxic_logreg = results_df.nlargest(10, 'prob_logreg')[['text', 'prob_logreg', 'toxic_actual']]
top_toxic_rf = results_df.nlargest(10, 'prob_rf')[['text', 'prob_rf', 'toxic_actual']]

print("\nТОП-10 самых токсичных текстов по LogisticRegression:")
print("-" * 80)
for i, (idx, row) in enumerate(top_toxic_logreg.iterrows(), 1):
    actual_label = "ТОКСИЧНЫЙ" if row['toxic_actual'] == 1 else "НЕ токсичный"
    prob_percent = row['prob_logreg'] * 100
    print(f"{i:2d}. [{prob_percent:5.1f}%] {row['text']}")
    print(f"     Фактический класс: {actual_label}")
    print()

print("\nТОП-10 самых токсичных текстов по RandomForest:")
print("-" * 80)
for i, (idx, row) in enumerate(top_toxic_rf.iterrows(), 1):
    actual_label = "ТОКСИЧНЫЙ" if row['toxic_actual'] == 1 else "НЕ токсичный"
    prob_percent = row['prob_rf'] * 100
    print(f"{i:2d}. [{prob_percent:5.1f}%] {row['text']}")
    print(f"     Фактический класс: {actual_label}")
    print()

# СРАВНИТЕЛЬНЫЙ АНАЛИЗ
print("\n" + "="*70)
print("СРАВНИТЕЛЬНЫЙ АНАЛИЗ РЕЗУЛЬТАТОВ")
print("="*70)

# Тексты, которые попали в топ-10 обоих классификаторов
common_texts = set(top_toxic_logreg['text']).intersection(set(top_toxic_rf['text']))
unique_logreg = set(top_toxic_logreg['text']) - set(top_toxic_rf['text'])
unique_rf = set(top_toxic_rf['text']) - set(top_toxic_logreg['text'])

print(f"Тексты, попавшие в топ-10 ОБОИХ классификаторов ({len(common_texts)}):")
print("-" * 60)
for i, text in enumerate(common_texts, 1):
    text_data = results_df[results_df['text'] == text].iloc[0]
    print(f"{i}. {text}")
    print(f"   LogisticRegression: {text_data['prob_logreg']:.3f}, RandomForest: {text_data['prob_rf']:.3f}")
    print()

print(f"Тексты, попавшие в топ-10 ТОЛЬКО LogisticRegression ({len(unique_logreg)}):")
print("-" * 60)
for i, text in enumerate(unique_logreg, 1):
    text_data = results_df[results_df['text'] == text].iloc[0]
    actual_label = "ТОКСИЧНЫЙ" if text_data['toxic_actual'] == 1 else "НЕ токсичный"
    print(f"{i}. {text}")
    print(f"   Вероятность: {text_data['prob_logreg']:.3f}, Фактический: {actual_label}")
    print()

print(f"Тексты, попавшие в топ-10 ТОЛЬКО RandomForest ({len(unique_rf)}):")
print("-" * 60)
for i, text in enumerate(unique_rf, 1):
    text_data = results_df[results_df['text'] == text].iloc[0]
    actual_label = "ТОКСИЧНЫЙ" if text_data['toxic_actual'] == 1 else "НЕ токсичный"
    print(f"{i}. {text}")
    print(f"   Вероятность: {text_data['prob_rf']:.3f}, Фактический: {actual_label}")
    print()

# СТАТИСТИКА КАЧЕСТВА
print("\n" + "="*70)
print("СТАТИСТИКА КАЧЕСТВА ПРЕДСКАЗАНИЙ")
print("="*70)

logreg_top_accuracy = top_toxic_logreg['toxic_actual'].mean()
rf_top_accuracy = top_toxic_rf['toxic_actual'].mean()

print(f"Доля действительно токсичных в топ-10 LogisticRegression: {logreg_top_accuracy:.1%}")
print(f"Доля действительно токсичных в топ-10 RandomForest: {rf_top_accuracy:.1%}")

# Анализ важных признаков
print(f"\n" + "="*70)
print("АНАЛИЗ ВАЖНЫХ ПРИЗНАКОВ")
print("="*70)

# Для LogisticRegression
feature_names_count = count_vectorizer.get_feature_names_out()
logreg_coef = logreg.coef_[0]
top_logreg_features_idx = np.argsort(logreg_coef)[-15:]
top_logreg_features = [(feature_names_count[i], logreg_coef[i]) for i in top_logreg_features_idx]

print(" Топ-15 самых токсичных слов по LogisticRegression:")
for word, coef in reversed(top_logreg_features):
    print(f"   {word}: {coef:.4f}")

# Для RandomForest
feature_names_tfidf = tfidf_vectorizer.get_feature_names_out()
rf_feature_importance = rf.feature_importances_
top_rf_features_idx = np.argsort(rf_feature_importance)[-15:]
top_rf_features = [(feature_names_tfidf[i], rf_feature_importance[i]) for i in top_rf_features_idx]

print("\n Топ-15 самых важных слов по RandomForest:")
for word, importance in reversed(top_rf_features):
    print(f"   {word}: {importance:.4f}")




Размер датасета: 14412
Баланс классов:
toxic
0.0    9586
1.0    4826
Name: count, dtype: int64

Размер тренировочной выборки: 11529
Размер тестовой выборки: 2883
Баланс классов: 0 - 7668, 1 - 3861

ПАРА 1: LogisticRegression + CountVectorizer
F1-score LogisticRegression: 0.7759
              precision    recall  f1-score   support

         0.0       0.90      0.86      0.88      1918
         1.0       0.74      0.81      0.78       965

    accuracy                           0.84      2883
   macro avg       0.82      0.84      0.83      2883
weighted avg       0.85      0.84      0.84      2883


ПАРА 2: RandomForest + TfidfVectorizer


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['кое', 'нибудь'] not in stop_words.
  warnings.warn(


F1-score RandomForest: 0.6872
              precision    recall  f1-score   support

         0.0       0.87      0.75      0.81      1918
         1.0       0.61      0.79      0.69       965

    accuracy                           0.76      2883
   macro avg       0.74      0.77      0.75      2883
weighted avg       0.79      0.76      0.77      2883


ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ
LogisticRegression + CountVectorizer: F1 = 0.7759
RandomForest + TfidfVectorizer: F1 = 0.6872

АНАЛИЗ САМЫХ ТОКСИЧНЫХ ТЕКСТОВ

ТОП-10 самых токсичных текстов по LogisticRegression:
--------------------------------------------------------------------------------
 1. [100.0%] блеаадь как же обидно когда создаешь тред пародируешь речь ватников случайно употребив слово из скрипта и он скрывается у всех пересоздаю я много лет тут сижу с вами и обсераю пидорашек хоть я и сам один из них смеюсь над их смертями делаю фотожабы с обезьянами я то думал это делают русские русофобы и украинцы которых процентов 10 поэтому не от

## Задание 3 (4 балла - 1 балл за каждый классификатор)

Для классификаторов Logistic Regression, Decision Trees, Naive Bayes, RandomForest найдите способ извлечь важность признаков для предсказания токсичного класса. Сопоставьте полученные числа со словами (или нграммами) в словаре и найдите топ - 5 "токсичных" слов для каждого из классификаторов.

Важное требование: в топе не должно быть стоп-слов. Для этого вам нужно будет правильным образом настроить векторизацию.
Также как и в предыдущем задании у классификаторов должно быть задано вручную как минимум 2 параметра (по возможности, f1 мера каждого из классификаторов должна быть минимум 0.75

In [20]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import re

# Загрузка данных
data = pd.read_csv('/content/drive/MyDrive/labeled.csv', sep=',')
print(f"Размер датасета: {len(data)}")
print(f"Баланс классов:\n{data['toxic'].value_counts()}")

# Русские стоп-слова для фильтрации
russian_stopwords = [
    'и', 'в', 'на', 'с', 'по', 'у', 'о', 'к', 'а', 'но', 'это', 'вот',
    'ну', 'да', 'нет', 'же', 'ли', 'бы', 'то', 'меня', 'тебя', 'он',
    'она', 'они', 'мы', 'вы', 'мне', 'тебе', 'ему', 'ей', 'нам', 'вам',
    'мной', 'тобой', 'им', 'ею', 'нами', 'вами', 'мой', 'твой', 'его',
    'её', 'наш', 'ваш', 'их', 'свой', 'кто', 'что', 'как', 'где', 'когда',
    'почему', 'зачем', 'какой', 'который', 'сколько', 'чей', 'чтобы',
    'если', 'хотя', 'потому', 'так', 'тоже', 'или', 'ибо', 'чем', 'не',
    'ни', 'без', 'до', 'из', 'от', 'при', 'через', 'для', 'про', 'со',
    'тот', 'этот', 'такой', 'какой-то', 'кое-какой', 'какой-нибудь',
    'некий', 'некоторый', 'всякий', 'каждый', 'любой', 'самый', 'другой',
    'иной', 'сам', 'сама', 'само', 'сами', 'весь', 'вся', 'всё', 'все',
    'самого', 'самой', 'самом', 'самих', 'всего', 'всей', 'всём', 'всех',
    'есть', 'можно', 'очень', 'например', 'было'
]

# Предобработка текста
def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'[^\w\s!?]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

data['comment_clean'] = data['comment'].apply(preprocess_text)

# Разделение на train/test
X = data['comment_clean']
y = data['toxic']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nРазмер тренировочной выборки: {len(X_train)}")
print(f"Размер тестовой выборки: {len(X_test)}")

# Общий векторизатор для всех моделей (без стоп-слов)
vectorizer = CountVectorizer(
    max_features=8000,
    min_df=2,
    max_df=0.85,
    ngram_range=(1, 2),
    stop_words=russian_stopwords,
    analyzer='word'
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

feature_names = vectorizer.get_feature_names_out()
print(f"Количество признаков после фильтрации стоп-слов: {len(feature_names)}")

# Функция для получения топ-5 токсичных слов
def get_top_toxic_words(model, feature_names, model_name):
    if hasattr(model, 'coef_'):  # LogisticRegression
        importance = model.coef_[0]
    elif hasattr(model, 'feature_importances_'):  # DecisionTree, RandomForest
        importance = model.feature_importances_
    elif hasattr(model, 'feature_log_prob_'):  # NaiveBayes
        importance = model.feature_log_prob_[1, :] - model.feature_log_prob_[0, :]
    else:
        return []

    # Сортируем по важности для токсичного класса
    top_indices = np.argsort(importance)[-10:]
    top_words = []

    for idx in reversed(top_indices):
        word = feature_names[idx]
        if word not in russian_stopwords and len(word) > 2:
            top_words.append((word, importance[idx]))
            if len(top_words) >= 5:
                break

    return top_words

# 1. Logistic Regression
print("\n" + "="*70)
print("1. LOGISTIC REGRESSION")
print("="*70)

logreg = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

logreg.fit(X_train_vec, y_train)
y_pred_logreg = logreg.predict(X_test_vec)
f1_logreg = f1_score(y_test, y_pred_logreg)

print(f"F1-score: {f1_logreg:.4f}")
print(classification_report(y_test, y_pred_logreg))

# Топ-5 токсичных слов для Logistic Regression
top_logreg_words = get_top_toxic_words(logreg, feature_names, "LogisticRegression")
print("\n🔤 ТОП-5 токсичных слов (LogisticRegression):")
for i, (word, importance) in enumerate(top_logreg_words, 1):
    print(f"   {i}. '{word}': {importance:.4f}")

# 2. Decision Tree
print("\n" + "="*70)
print("2. DECISION TREE")
print("="*70)

dt = DecisionTreeClassifier(
    max_depth=20,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42
)

dt.fit(X_train_vec, y_train)
y_pred_dt = dt.predict(X_test_vec)
f1_dt = f1_score(y_test, y_pred_dt)

print(f"F1-score: {f1_dt:.4f}")
print(classification_report(y_test, y_pred_dt))

# Топ-5 токсичных слов для Decision Tree
top_dt_words = get_top_toxic_words(dt, feature_names, "DecisionTree")
print("\n🔤 ТОП-5 токсичных слов (Decision Tree):")
for i, (word, importance) in enumerate(top_dt_words, 1):
    print(f"   {i}. '{word}': {importance:.4f}")

# 3. Naive Bayes
print("\n" + "="*70)
print("3. NAIVE BAYES")
print("="*70)

nb = MultinomialNB(
    alpha=1.0,
    fit_prior=True
)

nb.fit(X_train_vec, y_train)
y_pred_nb = nb.predict(X_test_vec)
f1_nb = f1_score(y_test, y_pred_nb)

print(f"F1-score: {f1_nb:.4f}")
print(classification_report(y_test, y_pred_nb))

# Топ-5 токсичных слов для Naive Bayes
top_nb_words = get_top_toxic_words(nb, feature_names, "NaiveBayes")
print("\n🔤 ТОП-5 токсичных слов (Naive Bayes):")
for i, (word, importance) in enumerate(top_nb_words, 1):
    print(f"   {i}. '{word}': {importance:.4f}")

# 4. Random Forest
print("\n" + "="*70)
print("4. RANDOM FOREST")
print("="*70)

rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=25,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_vec, y_train)
y_pred_rf = rf.predict(X_test_vec)
f1_rf = f1_score(y_test, y_pred_rf)

print(f"F1-score: {f1_rf:.4f}")
print(classification_report(y_test, y_pred_rf))

# Топ-5 токсичных слов для Random Forest
top_rf_words = get_top_toxic_words(rf, feature_names, "RandomForest")
print("\n🔤 ТОП-5 токсичных слов (Random Forest):")
for i, (word, importance) in enumerate(top_rf_words, 1):
    print(f"   {i}. '{word}': {importance:.4f}")

# СВОДНАЯ ТАБЛИЦА ТОП-5 ТОКСИЧНЫХ СЛОВ
print("\n" + "="*70)
print("СВОДНАЯ ТАБЛИЦА: ТОП-5 ТОКСИЧНЫХ СЛОВ ПО КЛАССИФИКАТОРАМ")
print("="*70)

# Создаем сводную таблицу
summary_data = []
max_words = max(len(top_logreg_words), len(top_dt_words), len(top_nb_words), len(top_rf_words))

for i in range(max_words):
    row = []
    for top_words in [top_logreg_words, top_dt_words, top_nb_words, top_rf_words]:
        if i < len(top_words):
            word, importance = top_words[i]
            row.append(f"{word} ({importance:.3f})")
        else:
            row.append("-")
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data,
                         columns=['LogisticRegression', 'DecisionTree', 'NaiveBayes', 'RandomForest'])

print(summary_df.to_string(index=True))

# АНАЛИЗ СОВПАДЕНИЙ
print("\n" + "="*70)
print("АНАЛИЗ СОВПАДЕНИЙ МЕЖДУ КЛАССИФИКАТОРАМИ")
print("="*70)

# Собираем все уникальные токсичные слова
all_toxic_words = set()
for top_words in [top_logreg_words, top_dt_words, top_nb_words, top_rf_words]:
    for word, _ in top_words:
        all_toxic_words.add(word)

print(f"Всего уникальных токсичных слов в топ-5: {len(all_toxic_words)}")

# Анализ частоты встречаемости слов
word_frequency = {}
for top_words in [top_logreg_words, top_dt_words, top_nb_words, top_rf_words]:
    for word, _ in top_words:
        word_frequency[word] = word_frequency.get(word, 0) + 1

print("\nЧастота встречаемости слов в топ-5:")
for word, freq in sorted(word_frequency.items(), key=lambda x: x[1], reverse=True):
    models = []
    if any(word in dict(top_logreg_words) for word, _ in top_logreg_words):
        models.append("LR")
    if any(word in dict(top_dt_words) for word, _ in top_dt_words):
        models.append("DT")
    if any(word in dict(top_nb_words) for word, _ in top_nb_words):
        models.append("NB")
    if any(word in dict(top_rf_words) for word, _ in top_rf_words):
        models.append("RF")

    print(f"  '{word}': {freq} раз(а) [{' '.join(models)}]")

# Слова, которые все модели считают токсичными
common_words = [word for word, freq in word_frequency.items() if freq >= 3]
if common_words:
    print(f"\n🎯 Слова, которые большинство моделей считают токсичными: {', '.join(common_words)}")

# ПРОВЕРКА ВЫПОЛНЕНИЯ ТРЕБОВАНИЙ
print("\n" + "="*70)
print("ПРОВЕРКА ВЫПОЛНЕНИЯ ТРЕБОВАНИЙ")
print("="*70)

# Проверка F1-score
f1_scores = {
    'LogisticRegression': f1_logreg,
    'DecisionTree': f1_dt,
    'NaiveBayes': f1_nb,
    'RandomForest': f1_rf
}

print("F1-score по классификаторам:")
for model, score in f1_scores.items():
    status = "✓" if score >= 0.75 else "✗"
    print(f"  {status} {model}: {score:.4f}")


# Проверка отсутствия стоп-слов
print("\nПроверка отсутствия стоп-слов в топ-5:")
all_top_words = []
for top_words in [top_logreg_words, top_dt_words, top_nb_words, top_rf_words]:
    all_top_words.extend([word for word, _ in top_words])

stopwords_in_top = [word for word in all_top_words if word in russian_stopwords]
print(stopwords_in_top)

# ДОПОЛНИТЕЛЬНЫЙ АНАЛИЗ: Примеры текстов с токсичными словами
print("\n" + "="*70)
print("ПРИМЕРЫ ТЕКСТОВ С ТОКСИЧНЫМИ СЛОВАМИ")
print("="*70)

# Находим примеры текстов, содержащих топ-токсичные слова
toxic_examples = {}
for word in all_top_words[:10]:  # первые 10 уникальных слов
    examples = data[data['comment_clean'].str.contains(word, na=False) & (data['toxic'] == 1)]['comment'].head(2)
    if len(examples) > 0:
        toxic_examples[word] = examples.tolist()

print("Примеры токсичных комментариев с выявленными словами:")
for word, examples in list(toxic_examples.items())[:5]:
    print(f"\nСлово: '{word}'")
    for i, example in enumerate(examples, 1):
        print(f"  {i}. {example}")

Размер датасета: 14412
Баланс классов:
toxic
0.0    9586
1.0    4826
Name: count, dtype: int64

Размер тренировочной выборки: 11529
Размер тестовой выборки: 2883


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['кое', 'нибудь'] not in stop_words.
  warnings.warn(


Количество признаков после фильтрации стоп-слов: 8000

1. LOGISTIC REGRESSION
F1-score: 0.7597
              precision    recall  f1-score   support

         0.0       0.89      0.85      0.87      1918
         1.0       0.72      0.80      0.76       965

    accuracy                           0.83      2883
   macro avg       0.81      0.82      0.81      2883
weighted avg       0.84      0.83      0.83      2883


🔤 ТОП-5 токсичных слов (LogisticRegression):
   1. 'хохлов': 3.1788
   2. 'хохлы': 2.8917
   3. 'дебил': 2.7166
   4. 'быдло': 2.2047
   5. 'сжечь': 2.1707

2. DECISION TREE
F1-score: 0.5598
              precision    recall  f1-score   support

         0.0       0.85      0.37      0.52      1918
         1.0       0.41      0.87      0.56       965

    accuracy                           0.54      2883
   macro avg       0.63      0.62      0.54      2883
weighted avg       0.71      0.54      0.53      2883


🔤 ТОП-5 токсичных слов (Decision Tree):
   1. 'хохлы': 0.0